# AI Lab - Practical Question No. 3
## Model-Based Reflex Agent: 2x2 Grid Simulation

- **Syllabus Area**: Intelligent Agents
- **Difficulty**: 🟡 Intermediate
- **Practical Type**: Implementation
- **Core AI Concept**: Model-Based Reflex Agent, Internal State Representation, Transition Model, Sensor Model
- **Python Concepts Tested**: Object-Oriented Programming (OOP), Sets (`set`), 2D Lists (`List[List[str]]`)
- **Complexity**: Time Complexity: $O(N)$ | Space Complexity: $O(N)$ where $N=4$ for 2x2 grid

---

### Problem Statement
Design a **Model-Based Reflex Agent** in Python capable of cleaning a 2x2 grid environment.
Unlike a simple reflex agent, this agent maintains an internal state variable to track which rooms it has already visited and cleaned, stopping execution automatically once the internal model confirms the entire grid is clean.

**Task Requirements**:
1. Create a class `ModelBasedAgent`.
2. Maintain an internal `visited` set.
3. Update the internal state based on incoming percepts and a transition model.
4. Terminate when the internal model indicates all accessible rooms are clean.

**Sample Input**: `[['Dirty', 'Clean'], ['Dirty', 'Dirty']]`  
**Sample Output**: `Agent visited all rooms. Cleaning complete. Shutting down.`

### Agent PEAS Specification

| PEAS Component | Specification for Question 3 |
| :--- | :--- |
| **P**erformance Measure | Clean all accessible rooms; terminate immediately once internal model confirms cleanliness; minimize redundant moves |
| **E**nvironment | 2x2 discrete grid of rooms: `(0,0)`, `(0,1)`, `(1,0)`, `(1,1)` with physical boundary walls |
| **A**ctuators | `Clean` (suck dirt), `Up`, `Down`, `Left`, `Right` (directional navigation), `Shutdown` |
| **S**ensors | Local coordinate and status detector: returns `((row, col), status)` |

### Internal Architecture of Model-Based Reflex Agent

$$\text{Action} = f(\text{Internal State}, \text{Current Percept})$$

1. **Internal Visited Set** (`self.visited`): A Python `set` storing tuples of visited coordinates `{(0, 0), (0, 1), ...}`.
2. **Internal World Model** (`self.model`): A dictionary mapping `(row, col) -> 'Clean' | 'Dirty'`.
3. **Transition Model**: Updates beliefs based on actions: $f(\text{State}, \text{Action}) \to \text{State}'$.
   - Action `Clean` updates internal model at current position to `Clean`.
   - Directional moves update coordinate positions within bounds.
4. **Sensor Model**: Maps physical state to percepts: $h(\text{State}) \to \text{Percept}$ (with optional sensor noise/failure).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running from notebooks/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.model_based_agent import (
    ModelBasedAgent,
    TwoByTwoGridEnvironment,
    run_model_based_simulation,
    STATUS_CLEAN,
    STATUS_DIRTY,
    ACTION_CLEAN,
    ACTION_SHUTDOWN,
    TERMINATION_MESSAGE,
)
print("Model-Based Reflex Agent components loaded successfully.")

## 1. Simulation: Sample Scenario

- **Grid State**: `[['Dirty', 'Clean'], ['Dirty', 'Dirty']]`
- **Agent Start Position**: `(0, 0)`
- **Expected Outcome**: Agent cleans room (0, 0), navigates through unvisited rooms (1, 0) and (1, 1), cleans them, reaches (0, 1) which is already clean, confirms the whole grid is clean via its internal model, and halts with the termination message.

In [ ]:
sample_grid = [
    [STATUS_DIRTY, STATUS_CLEAN],
    [STATUS_DIRTY, STATUS_DIRTY],
]

env = TwoByTwoGridEnvironment(initial_grid=sample_grid, start_pos=(0, 0))
agent = ModelBasedAgent(grid_dimensions=(2, 2), initial_pos=(0, 0))

result = run_model_based_simulation(env, agent)

print("=" * 75)
print("EXECUTION STEP TRACE:")
print("=" * 75)
for record in result.history:
    print(record.format_trace())

print("\n" + "-" * 75)
print(f"Sample Output: {result.termination_message}")
print("-" * 75)
print(result.summary())

## 2. Inspecting the Internal State & Model Dynamics

Let us trace how the agent's internal `visited` set and `model` dictionary evolve compared to the actual physical environment across steps.

In [ ]:
print(f"{'Step':<5} | {'Agent Pos':<10} | {'Action':<10} | {'Internal Visited Set':<25} | {'Internal Model State'}")
print("-" * 85)
for r in result.history:
    visited_str = str(sorted(r.internal_visited))
    model_str = str(r.internal_model)
    print(f"{r.step_number:<5} | {str(r.position):<10} | {r.action:<10} | {visited_str:<25} | {model_str}")

## 3. Optional Challenge: Sensor Failure Simulation

What happens when the agent's sensors are noisy?  
We introduce a **sensor failure probability** $p = 0.5$ where a physically `Dirty` room is falsely perceived as `Clean`.
This illustrates why understanding **Sensor Models** and **Belief State Uncertainty** is a crucial topic in advanced AI.

In [ ]:
faulty_env = TwoByTwoGridEnvironment(
    initial_grid=[
        [STATUS_DIRTY, STATUS_CLEAN],
        [STATUS_DIRTY, STATUS_DIRTY],
    ],
    start_pos=(0, 0),
    sensor_failure_prob=0.5,
    seed=42,
)
faulty_agent = ModelBasedAgent(grid_dimensions=(2, 2), initial_pos=(0, 0))

faulty_result = run_model_based_simulation(faulty_env, faulty_agent)

print("Trace with 50% Sensor Failure Probability:")
for rec in faulty_result.history:
    print(rec.format_trace())

print("\n--- Analysis of Sensor Error ---")
print(f"Agent Internal Model Believes Clean: {faulty_result.is_all_clean_model}")
print(f"Actual Physical Environment Clean:   {faulty_result.is_all_clean_actual}")
print(f"Physical Final Grid:                 {faulty_result.final_grid}")

## 4. Viva Voce Reference & Study Guide

### 1. Internal State vs. Environment State
- **Environment State**: The true, ground-truth physical reality of the external world (e.g., whether grid cell `(1, 0)` is actually dirty). The agent cannot access it directly—only through percepts.
- **Internal State**: The agent's subjective internal memory and belief state (`self.visited` and `self.model`), maintained over time. In partially observable environments, the internal state allows the agent to remember rooms visited earlier.
- **Key Distinction**: A simple reflex agent has zero internal state, so it cannot remember which rooms it has already cleaned and cannot know when the entire grid is clean. A model-based agent maintains internal state to make informed, goal-directed decisions.

### 2. Transition Models
- **Definition**: The agent's internal model of how the world changes: $f(\text{State}, \text{Action}) \to \text{State}'$.
- **Actuator Modeling**: After executing `Clean`, the transition model updates the internal record of that cell to `Clean` without needing to re-sense.
- **Kinematic Modeling**: When executing `Right`, the transition model computes the predicted next coordinate `(r, c + 1)` and respects grid boundaries.

### 3. Sensor Models
- **Definition**: The mathematical or computational mapping from true physical states to percepts: $h(\text{State}) \to \text{Percept}$ or $P(\text{Percept} \mid \text{State})$.
- **Sensor Failure / Noise**: In real-world robotics, sensors are imperfect (e.g., dirt camera obscured by dust). If a sensor fails, the agent's internal model might temporarily diverge from ground truth unless fault-tolerant sensing (probabilistic belief states) is employed.

### 4. Complexity
- **Time Complexity**: $O(N)$ where $N$ is total grid rooms ($N = 4$ for 2x2 grid). Each room is visited and cleaned in a bounded number of actions.
- **Space Complexity**: $O(N)$ auxiliary memory to store coordinates in `self.visited` set and key-value pairs in `self.model`.